# YOLOv8 Ball Detector - Kaggle Training

This notebook sets up and runs the ball detector training pipeline on Kaggle with GPU acceleration.

## Prerequisites
1. Upload your dataset to Kaggle as a dataset
2. Add the dataset to this notebook (+ Add Data button)
3. Enable GPU: Settings → Accelerator → GPU T4 x2

## Dataset Structure
Your Kaggle dataset should have this structure:
```
your-dataset/
├── images/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
└── labels/
    ├── image1.txt
    ├── image2.txt
    └── ...
```

## 1. Clone Repository

In [ ]:
# Clone your repository and setup Python path
import sys
import os

!git clone https://github.com/conalhughes/RE-ObjectDetector-NSFR.git
%cd RE-ObjectDetector-NSFR

# Add to Python path so modules are importable
if '/kaggle/working/RE-ObjectDetector-NSFR' not in sys.path:
    sys.path.insert(0, '/kaggle/working/RE-ObjectDetector-NSFR')

print("✓ Repository cloned and Python path configured")


## 2. Check GPU Availability

In [ ]:
# Verify GPU is available
!nvidia-smi

## 3. Setup Kaggle Dataset

**IMPORTANT:** Replace `your-dataset-name` with your actual Kaggle dataset name!

In [ ]:
# Setup data from Kaggle dataset
# Replace 'your-dataset-name' with your actual dataset name
!python setup_kaggle_data.py /kaggle/input/your-dataset-name

# Or let it auto-detect if you only have one dataset
# !python setup_kaggle_data.py

## 4. Verify Data Setup

In [ ]:
# Check that raw_data is setup correctly
import os

if os.path.exists('raw_data/images'):
    image_count = len([f for f in os.listdir('raw_data/images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"✓ Images: {image_count}")
else:
    print("✗ raw_data/images not found")

if os.path.exists('raw_data/labels'):
    label_count = len([f for f in os.listdir('raw_data/labels') if f.endswith('.txt')])
    print(f"✓ Labels: {label_count}")
else:
    print("✗ raw_data/labels not found")

## 5.5 Configure Object Classes

Configure the object classes for your detection task. Modify the classes below to match your dataset.


In [ ]:
# Load and display current class configuration
import yaml

with open('classes_config.yaml', 'r') as f:
    current_config = yaml.safe_load(f)

print(f"📊 Current Dataset: {current_config['dataset_name']}")
print(f"📋 Configured Classes ({len(current_config['classes'])}):")
for cls in current_config['classes']:
    print(f"   [{cls['id']}] {cls['name']:20s} - RGB{tuple(cls['color'])}")


In [ ]:
# ==============================================================================
# MODIFY CLASSES HERE
# ==============================================================================
# Edit the classes below to match your dataset
# Each class needs: id, name, and RGB color [0-255]

# OPTION 1: Single Class (Ball Detection) - DEFAULT
classes_config = {
    'dataset_name': 'Ball Detector',
    'classes': [
        {'id': 0, 'name': 'ball', 'color': [0, 255, 0]}
    ]
}

# OPTION 2: Multi-Class RoboCup
# Uncomment to use:
# classes_config = {
#     'dataset_name': 'RoboCup Multi-Object Detector',
#     'classes': [
#         {'id': 0, 'name': 'ball', 'color': [0, 255, 0]},        # Green
#         {'id': 1, 'name': 'robot', 'color': [255, 0, 0]},  # Red
#         {'id': 2, 'name': 'human', 'color': [0, 0, 255]}, # Blue
#         {'id': 3, 'name': 'L-Intersection', 'color': [255, 255, 0]},  # Cyan
#         {'id': 4, 'name': 'T-Intersection', 'color': [0, 255, 255]}, # Yellow
#         {'id': 5, 'name': 'X-Intersection', 'color': [255, 0, 255]},  # Magenta
#     ]
# }
# ==============================================================================
# Save configuration
# ==============================================================================
with open('classes_config.yaml', 'w') as f:
    yaml.dump(classes_config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Classes saved!")
print(f"  Dataset: {classes_config['dataset_name']}")
print(f"  Classes: {[c['name'] for c in classes_config['classes']]}")


In [ ]:
# Verify configuration loads correctly
import importlib
import config as cfg

# Reload config to apply changes
importlib.reload(cfg)

print("="*70)
print("✓ CONFIGURATION VERIFIED")
print("="*70)
print(f"\nDataset: {cfg.DATASET_NAME}")
print(f"Number of Classes: {cfg.NUM_CLASSES}")
print(f"Class Names: {cfg.CLASS_NAMES}")
print(f"Class Colors: {cfg.CLASS_COLORS}")
print("\nClass Mapping:")
for cls_id, cls_name in enumerate(cfg.CLASS_NAMES):
    print(f"  Class {cls_id}: {cls_name}")
print("\n" + "="*70)
print("Ready to train! Classes are configured correctly.")
print("="*70)


## 5. Train Model

Now you can use the shell scripts just like locally!

In [ ]:
# Train with default settings (100 epochs)
!./train.sh --device cuda

# Or customize training parameters
# !./train.sh --epochs 50 --batch-size 32 --learning-rate 0.01 --device cuda

## 6. View Training Results

In [ ]:
# Display training plot
from IPython.display import Image, display
import glob

# Find the latest training results plot
plots = glob.glob('stats/*/training_results_*.png')
if plots:
    latest_plot = max(plots, key=os.path.getctime)
    print(f"Training results: {latest_plot}")
    display(Image(filename=latest_plot))
else:
    print("No training plots found yet")

## 7. Test Model

In [ ]:
# Test the trained model
!./test.sh --device cuda

## 8. Download Trained Model

Download your trained model to use locally or in your robot!

In [ ]:
# List available models
!ls -lh models/*_best.pt

# The model file will be in the output section
# Click the folder icon in the right sidebar to download it

## Tips for Kaggle

### GPU Limits
- Free tier: 30 hours/week of GPU time
- Sessions timeout after 12 hours
- Use `--epochs 50` or `--patience 20` for faster training

### Saving Results
- Models in `models/` directory are saved in notebook output
- Click "Save Version" to preserve your work
- Download `*_best.pt` file for use elsewhere

### Memory Management
- Kaggle notebooks have ~30GB disk space
- This setup uses symlinks to save space
- Processed data in `data/` uses actual disk space

### Faster Training
```python
!./train.sh --epochs 50 --batch-size 32 --model-size n --device cuda
```

### Custom Configuration
```python
!./train.sh --epochs 100 --learning-rate 0.001 --device cuda
```